<a href="https://colab.research.google.com/github/Saliyah-53/saliyah-stanceeval2026/blob/main/negative_results/SaliAI_ALLaM_QLoRA.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# SaliAI — ALLaM-7B QLoRA Fine-Tuning (Track 2)

A research experiment for the paper: **QLoRA fine-tuning** of ALLaM vs.\ few-shot
prompting.

> The goal is not necessarily a higher score, but a scientific result: does light
> fine-tuning beat few-shot prompting on unseen targets?

- QLoRA = 4-bit training + small LoRA adapters (runs on a T4)
- The 7B base model is frozen; only ~0.1% of parameters are trained
- **Before running:** Runtime > Change runtime type > T4 GPU

In [ ]:
# 1) Install dependencies
!pip install -q -U transformers peft bitsandbytes accelerate trl datasets

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.6/11.6 MB 122.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.9/40.9 MB 21.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 885.0/885.0 kB 57.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 555.1/555.1 kB 45.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.1/50.1 MB 19.9 MB/s eta 0:00:00


In [ ]:
# 2) Check GPU
import torch
assert torch.cuda.is_available(), "فعّل GPU: Runtime > Change runtime type > T4 GPU"
print("GPU:", torch.cuda.get_device_name(0))

GPU: Tesla T4


In [ ]:
# 3) Configuration
import os
BASE_DIR="."; DATA_DIR=f"{BASE_DIR}/data"
TRAIN_PATH=f"{DATA_DIR}/train_track_2.csv"   # هدفان مرئيان
TEST_PATH =f"{DATA_DIR}/test_unseen.csv"     # الأهداف غير المرئية (644)
OUTPUT_NAME="submission_unseen_lora"
OUT_DIR=f"{BASE_DIR}/allam_lora_outputs"; os.makedirs(OUT_DIR, exist_ok=True)

TEXT_COL="text"; TARGET_COL="target"; LABEL_COL="stance"
MODEL_ID="humain-ai/ALLaM-7B-Instruct-preview"

MAX_STEPS=300          # خطوات تدريب محدودة (T4) — ارفعها لو عندك وقت
LR=2e-4; BATCH=2; GRAD_ACCUM=8   # الدفعة الفعلية = 16

TARGET_DESC={
 "Ecars":"السيارات الكهربائية والتحول إليها",
 "Trimester":"نظام الفصول الدراسية الثلاثة في التعليم",
 "Women Driving":"قيادة المرأة للسيارة",
 "Covid Vaccine":"لقاح كورونا","Digital Transformation":"التحول الرقمي",
 "Women empowerment":"تمكين المرأة",
}
def desc(t): t=str(t).strip(); return TARGET_DESC.get(t,t)
SYSTEM=("أنت مصنّف مواقف عربي دقيق. حدّد موقف كاتب التغريدة تجاه الهدف. "
        "أجب بكلمة إنجليزية واحدة فقط: Favor أو Against أو None.")

In [ ]:
# 4) Load ALLaM in 4-bit (QLoRA)
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig, prepare_model_for_kbit_training, get_peft_model
import torch

bnb=BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                       bnb_4bit_compute_dtype=torch.float16, bnb_4bit_use_double_quant=True)
tokenizer=AutoTokenizer.from_pretrained(MODEL_ID)
if tokenizer.pad_token is None: tokenizer.pad_token=tokenizer.eos_token
model=AutoModelForCausalLM.from_pretrained(MODEL_ID, quantization_config=bnb, device_map="auto")
model=prepare_model_for_kbit_training(model)

lora=LoraConfig(r=16, lora_alpha=32, lora_dropout=0.05, bias="none",
                task_type="CAUSAL_LM",
                target_modules=["q_proj","k_proj","v_proj","o_proj"])
model=get_peft_model(model, lora)
model.print_trainable_parameters()   # لازم يطبع ~0.1%

config.json:   0%|          | 0.00/686 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/1.62k [00:00<?, ?B/s]

tokenizer.model: reconstructing file:   0%|          |  0.00B / 1.23MB            

tokenizer.model: downloading bytes:           |  0.00B            

model.safetensors.index.json:   0%|          | 0.00/23.9k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/111 [00:00<?, ?B/s]

trainable params: 16,777,216 || all params: 7,017,336,832 || trainable%: 0.2391


In [ ]:
# 5) Format training data as chats (target + tweet -> stance)
import pandas as pd
from datasets import Dataset

df=pd.read_csv(TRAIN_PATH, keep_default_na=False)
if "tweet_text" in df.columns and TEXT_COL not in df.columns:
    df=df.rename(columns={"tweet_text":TEXT_COL})
df=df[(df[TEXT_COL]!="")&(df[LABEL_COL]!="")].copy()
print("train rows:", len(df), "| targets:", df[TARGET_COL].value_counts().to_dict())

def to_text(row):
    msgs=[{"role":"system","content":SYSTEM},
          {"role":"user","content":f"الهدف: {row[TARGET_COL]} ({desc(row[TARGET_COL])})\nالتغريدة: {row[TEXT_COL]}\nالموقف:"},
          {"role":"assistant","content":str(row[LABEL_COL]).strip()}]
    return tokenizer.apply_chat_template(msgs, tokenize=False)

ds=Dataset.from_dict({"text":[to_text(r) for _,r in df.iterrows()]})
print(ds[0]["text"][:400])

train rows: 2721 | targets: {'Covid Vaccine': 1373, 'Digital Transformation': 1348}
<s> [INST] <<SYS>>
أنت مصنّف مواقف عربي دقيق. حدّد موقف كاتب التغريدة تجاه الهدف. أجب بكلمة إنجليزية واحدة فقط: Favor أو Against أو None.
<</SYS>>

الهدف: Covid Vaccine (لقاح كورونا)
التغريدة:  روح حلل محد يم تطعيم كورونا شف الحرم البارح مليونين بدون شرط تحصين😅
الموقف: [/INST] None </s>


In [ ]:
# Fix BFloat16/fp16 conflict: cast trainable layers (LoRA + norm) to float32
import torch
for name, param in model.named_parameters():
    if param.requires_grad:
        param.data = param.data.float()
print("تم تحويل الطبقات المدرّبة لـ float32 ✓")

تم تحويل الطبقات المدرّبة لـ float32 ✓


In [ ]:
# 6) Training (QLoRA via TRL SFTTrainer)
from trl import SFTTrainer, SFTConfig

cfg = SFTConfig(
    output_dir=f"{OUT_DIR}/ckpt",
    per_device_train_batch_size=BATCH,
    gradient_accumulation_steps=GRAD_ACCUM,
    learning_rate=LR, max_steps=MAX_STEPS,
    warmup_ratio=0.05, logging_steps=20, save_steps=MAX_STEPS,
    bf16=False, fp16=False, report_to="none",
    dataset_text_field="text", packing=False,
    max_length=256,               # بدل max_seq_length (الاسم الجديد)
)
trainer = SFTTrainer(model=model, train_dataset=ds, args=cfg)
trainer.train()
model.save_pretrained(f"{OUT_DIR}/lora_adapter")
print("تم حفظ محوّل LoRA ✓")

[transformers] warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Adding EOS to train dataset:   0%|          | 0/2721 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/2721 [00:00<?, ? examples/s]

Building labels for train dataset:   0%|          | 0/2721 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/2721 [00:00<?, ? examples/s]

Dropping fully masked examples from train dataset:   0%|          | 0/2721 [00:00<?, ? examples/s]

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'pad_token_id': 2}.


Step,Training Loss
20,3.197921
40,1.595351
60,1.426859
80,1.475260
100,1.414248
120,1.418213
140,1.426409
160,1.391037
180,1.340234
200,1.319605


Step,Training Loss
20,3.197921
40,1.595351
60,1.426859
80,1.475260
100,1.414248
120,1.418213
140,1.426409
160,1.391037
180,1.340234
200,1.319605


تم حفظ محوّل LoRA ✓


In [ ]:
# 7) Inference on the test set with the fine-tuned model
import re
def parse_label(o):
    o=o.strip().lower()
    if "against" in o: return "Against"
    if "favor" in o or "support" in o: return "Favor"
    if "none" in o or "neutral" in o: return "None"
    if any(w in o for w in ["معارض","ضد","رفض"]): return "Against"
    if any(w in o for w in ["مؤيد","يؤيد","يدعم"]): return "Favor"
    return "None"

@torch.no_grad()
def classify(text, target):
    msgs=[{"role":"system","content":SYSTEM},
          {"role":"user","content":f"الهدف: {target} ({desc(target)})\nالتغريدة: {text}\nالموقف:"}]
    prompt=tokenizer.apply_chat_template(msgs, add_generation_prompt=True, tokenize=False)
    enc=tokenizer(prompt, return_tensors="pt", return_token_type_ids=False).to(model.device)
    out=model.generate(**enc, max_new_tokens=5, do_sample=False, pad_token_id=tokenizer.eos_token_id)
    return parse_label(tokenizer.decode(out[0][enc["input_ids"].shape[-1]:], skip_special_tokens=True))

from tqdm.auto import tqdm
te=pd.read_csv(TEST_PATH, keep_default_na=False)
if "tweet_text" in te.columns and TEXT_COL not in te.columns:
    te=te.rename(columns={"tweet_text":TEXT_COL})
preds=[classify(str(r[TEXT_COL]), str(r[TARGET_COL])) for _,r in tqdm(te.iterrows(), total=len(te))]
print("توزيع:", pd.Series(preds).value_counts().to_dict())

  0%|          | 0/644 [00:00<?, ?it/s]

[transformers] `use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`.
[transformers] Caching is incompatible with gradient checkpointing in LlamaDecoderLayer. Setting `past_key_values=None`.


توزيع: {'None': 363, 'Against': 281}


In [ ]:
# 8) Generate the submission file (Codabench)
import zipfile
assert len(preds)==len(te), "عدم تطابق الأسطر!"
txt=f"{OUT_DIR}/{OUTPUT_NAME}.txt"
open(txt,"w",encoding="utf-8").write("\n".join(preds)+"\n")
zp=f"{OUT_DIR}/{OUTPUT_NAME}.zip"
with zipfile.ZipFile(zp,"w",zipfile.ZIP_DEFLATED) as z: z.write(txt, arcname=f"{OUTPUT_NAME}.txt")
print(f"OK: {len(preds)} تنبؤ")
print("ارفع هذا:", zp)

OK: 644 تنبؤ
ارفع هذا: ./allam_lora_outputs/submission_unseen_lora.zip
